# Reference: Pipeline Walkthrough

> **This notebook is for reference only.** The actual pipeline runs via the CLI script:
> ```
> python scripts/run_region_batch.py --all --cross-region --finalize
> ```
> Use `notebooks/1.1-validate-edges.ipynb` to verify results.

This notebook documents what the pipeline does step-by-step, useful for
understanding the computation or debugging individual regions interactively.

# 0.0 Setup

In [1]:
import sys
import json
import time
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import requests

# Add project root to path
PROJECT_DIR = Path.cwd().parent
sys.path.insert(0, str(PROJECT_DIR))

from modules.coordinates import load_unified, list_regions, get_region_schools
from modules.sparse_edges import (
    build_region_edges,
    build_cross_region_edges,
    build_all_edges,
    _default_adjacent_regions,
)
from modules.inter_island import tag_island_group, tag_sea_separated
from modules.gcs_utils import (
    OUTPUT_DIR,
    UGNAY_EDGES_DIR,
    UGNAY_COORDINATES_DIR,
    UGNAY_METADATA_DIR,
    upload_parquet,
    upload_json,
)

print(f"Project directory: {PROJECT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

Project directory: /workspace/project_ugnay
Output directory: /workspace/project_ugnay/output


In [2]:
# Configuration
HAVERSINE_CUTOFF_KM = 30.0
ROAD_CUTOFF_M = 20_000  # 20 km
OSRM_URL = "http://osrm:5000/table/v1/driving/"
EDGES_DIR = OUTPUT_DIR / "edges"
EDGES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Haversine pre-filter: {HAVERSINE_CUTOFF_KM} km")
print(f"Road distance cutoff: {ROAD_CUTOFF_M / 1000:.0f} km")
print(f"OSRM endpoint: {OSRM_URL}")
print(f"Edges output: {EDGES_DIR}")

Haversine pre-filter: 30.0 km
Road distance cutoff: 20 km
OSRM endpoint: http://osrm:5000/table/v1/driving/
Edges output: /workspace/project_ugnay/output/edges


# 1.0 Verify OSRM connectivity

In [3]:
# Quick OSRM health check — two points in Manila
try:
    resp = requests.get(
        "http://osrm:5000/route/v1/driving/121.0,14.5;121.1,14.6",
        timeout=10,
    )
    data = resp.json()
    if data.get("code") == "Ok":
        dist_km = data["routes"][0]["distance"] / 1000
        print(f"OSRM is running. Test route: {dist_km:.1f} km")
    else:
        print(f"OSRM returned unexpected code: {data.get('code')}")
except requests.ConnectionError:
    print("ERROR: Cannot connect to OSRM at http://osrm:5000")
    print("Start it with: docker compose --profile routing up -d")
    raise

OSRM is running. Test route: 20.5 km


# 2.0 Load school coordinates

In [4]:
df_all = load_unified()

  Excluded 4,557 schools by coord_rejection_reason:
    no_submission: 2,385
    no_coordinate_source: 580
    invalid: 506
    placeholder_default: 504
    out_of_bounds: 203
    outside_all_polygons: 165
    not_in_lis: 156
    coordinate_cluster: 58
Loaded 55,864 schools with valid coordinates
  Public:  47,607
  Private: 8,257


In [ ]:
# Tag island groups
df_all = tag_island_group(df_all)

# Regional breakdown
region_counts = (
    df_all.groupby("region")
    .agg(
        total=("school_id", "count"),
        public=("sector", lambda s: (s == "public").sum()),
        private=("sector", lambda s: (s == "private").sum()),
        island=("island_group", "first"),
    )
    .sort_values("total", ascending=False)
)
print(region_counts.to_string())
print(f"\nTotal: {len(df_all):,} schools across {len(region_counts)} regions")

In [ ]:
# Save coordinate snapshot for reproducibility
snapshot_path = EDGES_DIR / "schools_unified_snapshot.parquet"
df_all.to_parquet(snapshot_path, index=False)
print(f"Saved coordinate snapshot: {snapshot_path.name} ({len(df_all):,} rows)")

# 3.0 Compute within-region edges

Run OSRM Table API for each region. Each region produces a separate Parquet file.

**Estimated time:** ~5 min per region, ~85 min total for 17 regions.

> If this is interrupted, you can skip completed regions by checking which files already exist in `output/edges/`. See Section 3.1 for selective re-runs.

In [ ]:
%%time

regions = list_regions(df_all)
region_results = {}

for region in regions:
    # Check if already computed (for crash recovery)
    safe_name = region.lower().replace(" ", "_").replace("-", "_")
    out_path = EDGES_DIR / f"region_{safe_name}.parquet"

    if out_path.exists():
        df_existing = pd.read_parquet(out_path)
        print(f"\nSkipping {region} — already exists ({len(df_existing):,} edges)")
        region_results[region] = len(df_existing)
        continue

    df_region = get_region_schools(df_all, region)

    t0 = time.time()
    df_edges = build_region_edges(
        df_region,
        haversine_cutoff_km=HAVERSINE_CUTOFF_KM,
        road_cutoff_m=ROAD_CUTOFF_M,
        osrm_url=OSRM_URL,
        region_name=region,
    )
    elapsed = time.time() - t0

    # Add metadata columns
    df_edges["source_region"] = region
    df_edges["is_cross_region"] = False

    # Save
    if len(df_edges) > 0:
        df_edges.to_parquet(out_path, index=False)
        print(f"  Saved: {out_path.name} ({len(df_edges):,} edges, {elapsed:.0f}s)")
    else:
        # Save empty file to mark as processed
        df_edges.to_parquet(out_path, index=False)
        print(f"  Saved: {out_path.name} (0 edges, {elapsed:.0f}s)")

    region_results[region] = len(df_edges)

print(f"\n{'='*60}")
print("Within-region summary:")
for r, n in sorted(region_results.items(), key=lambda x: -x[1]):
    print(f"  {r}: {n:,} edges")
print(f"  Total: {sum(region_results.values()):,} edges")

## 3.1 Selective re-run (optional)

If a specific region failed or needs recomputation, delete its parquet file and re-run this cell.

In [ ]:
# Uncomment and modify to re-run a specific region:
# RERUN_REGION = "Region IV-A"
# safe_name = RERUN_REGION.lower().replace(" ", "_").replace("-", "_")
# out_path = EDGES_DIR / f"region_{safe_name}.parquet"
# if out_path.exists():
#     out_path.unlink()
#     print(f"Deleted {out_path.name} — re-run Section 3.0 to recompute")

# 4.0 Compute cross-region boundary edges

For adjacent region pairs, find schools near the boundary and compute
OSRM distances between them. Only cross-region edges are retained.

**Estimated time:** ~30 min for all adjacent pairs.

In [ ]:
%%time

adjacent_pairs = _default_adjacent_regions()
cross_region_path = EDGES_DIR / "cross_region_pairs.parquet"

if cross_region_path.exists():
    df_cross_existing = pd.read_parquet(cross_region_path)
    print(f"Cross-region edges already exist: {len(df_cross_existing):,} edges")
    print("Delete the file to recompute.")
else:
    cross_edges = []

    for region_a, region_b in adjacent_pairs:
        # Skip if either region not in data
        if region_a not in regions or region_b not in regions:
            print(f"Skipping {region_a} ↔ {region_b} (region not in data)")
            continue

        t0 = time.time()
        df_cross = build_cross_region_edges(
            df_all, region_a, region_b,
            haversine_cutoff_km=HAVERSINE_CUTOFF_KM,
            road_cutoff_m=ROAD_CUTOFF_M,
            osrm_url=OSRM_URL,
        )
        elapsed = time.time() - t0

        if len(df_cross) > 0:
            df_cross["is_cross_region"] = True
            # Assign source_region based on source school's actual region
            src_regions = df_all.set_index("school_id")["region"]
            df_cross["source_region"] = df_cross["source_id"].map(src_regions)
            cross_edges.append(df_cross)
            print(f"  {region_a} ↔ {region_b}: {len(df_cross):,} edges ({elapsed:.0f}s)")
        else:
            print(f"  {region_a} ↔ {region_b}: 0 edges ({elapsed:.0f}s)")

    if cross_edges:
        df_cross_all = pd.concat(cross_edges, ignore_index=True)
        # Deduplicate (A→B and B→A pairs may overlap across adjacent pair computations)
        df_cross_all = df_cross_all.drop_duplicates(
            subset=["source_id", "target_id"], keep="first"
        )
        df_cross_all.to_parquet(cross_region_path, index=False)
        print(f"\nSaved: {cross_region_path.name} ({len(df_cross_all):,} edges)")
    else:
        print("\nNo cross-region edges found.")

# 5.0 Combine and tag all edges

In [ ]:
# Load all per-region parquets + cross-region
edge_files = sorted(EDGES_DIR.glob("region_*.parquet")) + list(EDGES_DIR.glob("cross_region_pairs.parquet"))
print(f"Loading {len(edge_files)} edge files:")
for f in edge_files:
    print(f"  {f.name}")

dfs = [pd.read_parquet(f) for f in edge_files]
df_edges = pd.concat(dfs, ignore_index=True)

# Tag sea-separated edges
df_edges = tag_sea_separated(df_edges)

print(f"\nCombined edge table: {len(df_edges):,} edges")
print(f"  Within-region: {(~df_edges['is_cross_region']).sum():,}")
print(f"  Cross-region:  {df_edges['is_cross_region'].sum():,}")
print(f"  Sea-separated: {df_edges['is_sea_separated'].sum():,}")
print(f"\nColumns: {list(df_edges.columns)}")
print(f"Memory: {df_edges.memory_usage(deep=True).sum() / 1e6:.1f} MB")

In [ ]:
# Quick sanity checks
print("Edge distance statistics (km):")
road_km = df_edges["road_distance_m"] / 1000
print(road_km.describe().to_string())

print(f"\nRoad-to-haversine ratio statistics:")
ratio = df_edges["road_haversine_ratio"].dropna()
print(ratio.describe().to_string())

# Edges per school
src_counts = df_edges.groupby("source_id").size()
print(f"\nEdges per source school:")
print(src_counts.describe().to_string())

# Schools with zero edges (isolated)
all_ids = set(df_all["school_id"])
connected_ids = set(df_edges["source_id"]) | set(df_edges["target_id"])
isolated = all_ids - connected_ids
print(f"\nSchools with no edges (isolated): {len(isolated):,} / {len(all_ids):,}")

# 6.0 Save combined edge table and manifest

In [ ]:
# Save combined edge table
combined_path = EDGES_DIR / "all_edges.parquet"
df_edges.to_parquet(combined_path, index=False)
print(f"Saved combined edges: {combined_path.name}")
print(f"  Size: {combined_path.stat().st_size / 1e6:.1f} MB")
print(f"  Rows: {len(df_edges):,}")

In [ ]:
# Generate manifest
manifest = {
    "version": "v1",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "parameters": {
        "haversine_cutoff_km": HAVERSINE_CUTOFF_KM,
        "road_cutoff_m": ROAD_CUTOFF_M,
        "osrm_profile": "car",
        "osrm_url": OSRM_URL,
    },
    "statistics": {
        "n_schools_total": len(df_all),
        "n_schools_public": int((df_all["sector"] == "public").sum()),
        "n_schools_private": int((df_all["sector"] == "private").sum()),
        "n_edges_total": len(df_edges),
        "n_edges_within_region": int((~df_edges["is_cross_region"]).sum()),
        "n_edges_cross_region": int(df_edges["is_cross_region"].sum()),
        "n_edges_sea_separated": int(df_edges["is_sea_separated"].sum()),
        "n_schools_isolated": len(isolated),
        "n_regions": len(regions),
        "regions": regions,
        "mean_edges_per_school": float(src_counts.mean()),
        "median_edges_per_school": float(src_counts.median()),
    },
    "files": {
        "combined": "all_edges.parquet",
        "per_region": [f.name for f in sorted(EDGES_DIR.glob("region_*.parquet"))],
        "cross_region": "cross_region_pairs.parquet",
        "coordinate_snapshot": "schools_unified_snapshot.parquet",
    },
}

manifest_path = EDGES_DIR / "_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)
print(f"Saved manifest: {manifest_path.name}")
print(json.dumps(manifest["statistics"], indent=2))

# 7.0 Upload to GCS

In [ ]:
# Upload all edge files to GCS
print("Uploading to GCS...")

# Per-region parquets + combined + cross-region
for f in sorted(EDGES_DIR.glob("*.parquet")):
    upload_parquet(f, str(UGNAY_EDGES_DIR))

# Manifest
upload_json(manifest_path, str(UGNAY_EDGES_DIR))

# Coordinate snapshot
upload_parquet(snapshot_path, str(UGNAY_COORDINATES_DIR))

print("\nAll files uploaded to GCS.")

# Summary

This walkthrough covers what `scripts/run_region_batch.py` does under the hood:
1. Loads unified coordinates from project_coordinates
2. Verifies OSRM connectivity
3. Computes within-region edges (batched OSRM Table API, haversine pre-filter)
4. Computes cross-region boundary edges for adjacent region pairs
5. Combines all edges, tags sea-separated pairs
6. Generates manifest with statistics
7. Uploads to GCS

**To run the pipeline:** `python scripts/run_region_batch.py --all --cross-region --finalize`
**To verify results:** `notebooks/1.1-validate-edges.ipynb`